**© Copyright AIDENTIFY. All rights reserved.**

# Part 4 | Session 06: 그래프 RAG - 지식 그래프 기반 검색 증강

## 학습 목표

1️⃣ 텍스트에서 엔티티/관계를 추출해 **지식 그래프**를 구축한다  
2️⃣ **Cypher 경로 패턴**을 파싱해 그래프를 탐색하는 시뮬레이터를 만든다  
3️⃣ 자연어 질문을 LLM 이 **Cypher 로 번역**하게 한다 (Text-to-Cypher)  
4️⃣ 검색된 트리플만 근거로 **LLM 답변**을 생성한다  

---

### 실습 환경
- **GPU**: 선택사항
- **필수 패키지**: networkx, openai, matplotlib
- **외부 서비스**: 없음 (그래프는 NetworkX 메모리 그래프로 다룹니다)

In [ ]:
# 💡 setup.sh 실행했으면 이 셀은 건너뛰세요 (참고용 — 본 노트북이 필요로 하는 패키지)
# 패키지 설치
# !pip install -q networkx matplotlib openai python-dotenv

In [ ]:
# 패키지 임포트 및 환경 설정
import os
import json
import networkx as nx
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple
from dotenv import load_dotenv, find_dotenv

# .env 로드 — OPENAI_API_KEY 를 환경변수로 (상위 폴더까지 탐색)
load_dotenv(find_dotenv(usecwd=True))

_have = lambda k: "✅" if os.environ.get(k) else "❌(미설정)"
print("패키지 로드 완료")
print(f"  OPENAI_API_KEY: {_have('OPENAI_API_KEY')}")

---

## 실습 흐름

```
① 그래프 구축      텍스트 → LLM 엔티티/관계 추출 → NetworkX 지식 그래프
② Cypher 시뮬레이터  경로 패턴 파싱 → 그래프 탐색 (Neo4j 서버 불필요)
③ 자연어 질문       "이재용이 회장인 그룹의 계열사 CEO 들은 어느 학교를 나왔나?"
④ Cypher 생성       LLM 이 스키마를 보고 질문을 Cypher 로 번역
⑤ 검색             생성된 Cypher 를 시뮬레이터로 실행
⑥ LLM 답변          검색된 트리플만 근거로 답변 생성
```

벡터 RAG 는 ③ 같은 멀티홉 질문에서 중간 단계(그룹·계열사)가 질문에 드러나지
않아 한 번의 유사도 검색으로 연결하지 못합니다. 그래프 RAG 는 그 연결을
**관계를 따라가는 탐색**으로 바꿉니다.


In [ ]:
# Step 1: 텍스트 → LLM 엔티티/관계 추출
from pathlib import Path
from openai import OpenAI

DATA_PATH = Path("data/companies.txt")
raw_text = DATA_PATH.read_text(encoding="utf-8")

# 주석/공백 제거
sentences = [s.strip() for s in raw_text.splitlines()
             if s.strip() and not s.startswith("#")]
print(f"문장 수: {len(sentences)}")
print("샘플:")
for s in sentences[:3]:
    print(f"  • {s}")
print()


EXTRACTION_PROMPT = """다음 한국어 텍스트에서 엔티티와 관계를 추출해 JSON 으로 반환하세요.

관계 타입(정확히 이 4가지만 사용):
  - HAS_CHAIRMAN   : 그룹의 회장
  - SUBSIDIARY_OF  : 계열사가 어느 그룹 소속인지
  - HAS_CEO        : 계열사의 CEO
  - GRADUATED_FROM : 인물이 졸업한 학교

엔티티 타입: 그룹, 계열사, 인물, 학교

⚠ 중요 규칙 (반드시 지킬 것):
1. 엔티티 이름(name)에 한국어 조사(은/는/이/가/을/를/의/에/와/과/도/로 등)를 절대 포함하지 마세요.
   예) "삼성전자의" → "삼성전자",  "삼성전자는" → "삼성전자"
2. relations 의 subject/object 는 entities 의 name 과 글자까지 완전히 동일해야 합니다.
3. 텍스트에 나오는 학교도 빠짐없이 entities 에 "학교" 타입으로 포함하세요.

형식:
{{
  "entities": [{{"name": "...", "type": "..."}}],
  "relations": [{{"subject": "...", "predicate": "...", "object": "..."}}]
}}

텍스트:
{text}
"""


def extract_with_llm(text: str) -> dict:
    """LLM 호출. API 키 없거나 실패 시 미리 준비한 MOCK_RESULT 사용."""
    try:
        client = OpenAI()
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system",
                 "content": "당신은 정확한 한국어 지식 그래프 추출기입니다."},
                {"role": "user", "content": EXTRACTION_PROMPT.format(text=text)},
            ],
            temperature=0,
            response_format={"type": "json_object"},
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"⚠ LLM 호출 실패 → MOCK_RESULT 사용 ({type(e).__name__})")
        return MOCK_RESULT


# ── 강의 진행용 MOCK 결과 (API 키 없어도 실습 가능) ─────────
MOCK_RESULT = {
    "entities": [
        # 그룹
        {"name": "삼성그룹", "type": "그룹"},
        {"name": "SK그룹", "type": "그룹"},
        {"name": "현대자동차그룹", "type": "그룹"},
        # 회장
        {"name": "이재용", "type": "인물"}, {"name": "최태원", "type": "인물"},
        {"name": "정의선", "type": "인물"},
        # 계열사
        {"name": "삼성전자", "type": "계열사"}, {"name": "삼성물산", "type": "계열사"},
        {"name": "삼성SDS", "type": "계열사"}, {"name": "SK하이닉스", "type": "계열사"},
        {"name": "SK이노베이션", "type": "계열사"}, {"name": "SK텔레콤", "type": "계열사"},
        {"name": "현대자동차", "type": "계열사"}, {"name": "기아", "type": "계열사"},
        {"name": "현대모비스", "type": "계열사"},
        # CEO
        {"name": "한종희", "type": "인물"}, {"name": "오세철", "type": "인물"},
        {"name": "이준희", "type": "인물"}, {"name": "곽노정", "type": "인물"},
        {"name": "박상규", "type": "인물"}, {"name": "유영상", "type": "인물"},
        {"name": "장재훈", "type": "인물"}, {"name": "송호성", "type": "인물"},
        {"name": "이규석", "type": "인물"},
        # 학교
        {"name": "서울대학교", "type": "학교"}, {"name": "인하대학교", "type": "학교"},
        {"name": "연세대학교", "type": "학교"}, {"name": "시카고대학교", "type": "학교"},
        {"name": "고려대학교", "type": "학교"}, {"name": "한양대학교", "type": "학교"},
    ],
    "relations": [
        # 그룹 → 회장
        {"subject": "삼성그룹", "predicate": "HAS_CHAIRMAN", "object": "이재용"},
        {"subject": "SK그룹", "predicate": "HAS_CHAIRMAN", "object": "최태원"},
        {"subject": "현대자동차그룹", "predicate": "HAS_CHAIRMAN", "object": "정의선"},
        # 계열사 → 그룹
        {"subject": "삼성전자", "predicate": "SUBSIDIARY_OF", "object": "삼성그룹"},
        {"subject": "삼성물산", "predicate": "SUBSIDIARY_OF", "object": "삼성그룹"},
        {"subject": "삼성SDS", "predicate": "SUBSIDIARY_OF", "object": "삼성그룹"},
        {"subject": "SK하이닉스", "predicate": "SUBSIDIARY_OF", "object": "SK그룹"},
        {"subject": "SK이노베이션", "predicate": "SUBSIDIARY_OF", "object": "SK그룹"},
        {"subject": "SK텔레콤", "predicate": "SUBSIDIARY_OF", "object": "SK그룹"},
        {"subject": "현대자동차", "predicate": "SUBSIDIARY_OF", "object": "현대자동차그룹"},
        {"subject": "기아", "predicate": "SUBSIDIARY_OF", "object": "현대자동차그룹"},
        {"subject": "현대모비스", "predicate": "SUBSIDIARY_OF", "object": "현대자동차그룹"},
        # 계열사 → CEO
        {"subject": "삼성전자", "predicate": "HAS_CEO", "object": "한종희"},
        {"subject": "삼성물산", "predicate": "HAS_CEO", "object": "오세철"},
        {"subject": "삼성SDS", "predicate": "HAS_CEO", "object": "이준희"},
        {"subject": "SK하이닉스", "predicate": "HAS_CEO", "object": "곽노정"},
        {"subject": "SK이노베이션", "predicate": "HAS_CEO", "object": "박상규"},
        {"subject": "SK텔레콤", "predicate": "HAS_CEO", "object": "유영상"},
        {"subject": "현대자동차", "predicate": "HAS_CEO", "object": "장재훈"},
        {"subject": "기아", "predicate": "HAS_CEO", "object": "송호성"},
        {"subject": "현대모비스", "predicate": "HAS_CEO", "object": "이규석"},
        # 인물 → 학교
        {"subject": "이재용", "predicate": "GRADUATED_FROM", "object": "서울대학교"},
        {"subject": "최태원", "predicate": "GRADUATED_FROM", "object": "시카고대학교"},
        {"subject": "정의선", "predicate": "GRADUATED_FROM", "object": "고려대학교"},
        {"subject": "한종희", "predicate": "GRADUATED_FROM", "object": "인하대학교"},
        {"subject": "오세철", "predicate": "GRADUATED_FROM", "object": "서울대학교"},
        {"subject": "이준희", "predicate": "GRADUATED_FROM", "object": "연세대학교"},
        {"subject": "곽노정", "predicate": "GRADUATED_FROM", "object": "서울대학교"},
        {"subject": "박상규", "predicate": "GRADUATED_FROM", "object": "서울대학교"},
        {"subject": "유영상", "predicate": "GRADUATED_FROM", "object": "서울대학교"},
        {"subject": "장재훈", "predicate": "GRADUATED_FROM", "object": "서울대학교"},
        {"subject": "송호성", "predicate": "GRADUATED_FROM", "object": "한양대학교"},
        {"subject": "이규석", "predicate": "GRADUATED_FROM", "object": "서울대학교"},
    ],
}


# 실행 (LLM 또는 mock)
result = extract_with_llm(raw_text)

# 타입별 카운트
from collections import Counter
ent_types = Counter(e["type"] for e in result["entities"])
rel_types = Counter(r["predicate"] for r in result["relations"])

print(f"엔티티 {len(result['entities'])}개:  " +
      ",  ".join(f"{t} {n}" for t, n in ent_types.most_common()))
print(f"관계   {len(result['relations'])}개:  " +
      ",  ".join(f"{t} {n}" for t, n in rel_types.most_common()))


In [ ]:
# Step 2: 추출 결과 → NetworkX 지식 그래프 구축

def build_knowledge_graph(extraction_result: Dict) -> nx.DiGraph:
    """엔티티/관계 dict → DiGraph."""
    G = nx.DiGraph()
    for entity in extraction_result["entities"]:
        G.add_node(entity["name"], type=entity["type"])
    for relation in extraction_result["relations"]:
        G.add_edge(
            relation["subject"], relation["object"],
            relation=relation["predicate"],
        )
    return G


kg = build_knowledge_graph(result)
print(f"지식 그래프 구축 완료")
print(f"  노드: {kg.number_of_nodes()}개   엣지: {kg.number_of_edges()}개")

# 타입별 노드 확인
from collections import defaultdict
by_type = defaultdict(list)
for n, d in kg.nodes(data=True):
    by_type[d.get("type", "?")].append(n)

print("\n노드 (타입별):")
for t in ["그룹", "계열사", "인물", "학교"]:
    print(f"  [{t}] ({len(by_type[t])}개)  " + ", ".join(by_type[t]))

# 관계 타입별 엣지 수
rel_counts = Counter(d["relation"] for _, _, d in kg.edges(data=True))
print("\n엣지 (관계별):")
for rel, n in rel_counts.most_common():
    print(f"  [{rel}]  {n}개")


In [ ]:
# Step 3: Cypher 시뮬레이터 정의
#
# Cypher 경로 패턴을 파싱해 NetworkX 그래프 위에서 실행한다.
# Neo4j 서버는 쓰지 않는다.
import re

NODE_RE = re.compile(
    r"\(\s*(\w+)?\s*(?::\s*([^\s){]+))?\s*"
    r"(?:\{\s*name\s*:\s*['\"]([^'\"]+)['\"]\s*\})?\s*\)")
REL_RE = re.compile(r"\[\s*:?\s*(\w+)?\s*\]")


def parse_cypher(cypher: str) -> Dict:
    """MATCH 선형 경로 패턴 + RETURN 을 노드 / 관계체인 / 투영으로 분해.

    (p:인물 {name:'이재용'})<-[:HAS_CHAIRMAN]-(g:그룹)
        → nodes  = [{var:p, label:인물, name:이재용}, {var:g, label:그룹, name:None}]
          chain  = [("HAS_CHAIRMAN", "in")]      # in = 엣지를 거슬러 올라감
    """
    text = " ".join(cypher.split())
    m = re.search(r"MATCH\s+(.*?)(?:\s+RETURN\s+(.*))?$", text, re.I)
    if not m:
        raise ValueError("MATCH 절을 찾지 못했습니다.")
    pattern, ret = m.group(1), m.group(2)

    nodes, spans = [], []
    for nm in NODE_RE.finditer(pattern):
        nodes.append({"var": nm.group(1), "label": nm.group(2), "name": nm.group(3)})
        spans.append(nm.span())
    if len(nodes) < 2:
        raise ValueError("노드가 2개 이상인 경로 패턴이 필요합니다.")

    # 노드와 노드 사이의 텍스트에서 관계 타입과 방향을 읽는다
    chain = []
    for i in range(len(nodes) - 1):
        conn = pattern[spans[i][1]:spans[i + 1][0]]
        rm = REL_RE.search(conn)
        if not rm or not rm.group(1):
            raise ValueError(f"관계 타입을 읽지 못했습니다: '{conn.strip()}'")
        if conn.lstrip().startswith("<-"):
            direction = "in"
        elif conn.rstrip().endswith("->"):
            direction = "out"
        else:
            raise ValueError(f"방향(-> 또는 <-)이 없습니다: '{conn.strip()}'")
        chain.append((rm.group(1), direction))

    returns = []
    if ret:
        for item in re.split(r",(?![^(]*\))", ret):
            item = re.sub(r"\s+AS\s+\w+", "", item.strip(), flags=re.I)
            var, _, prop = item.partition(".")
            returns.append((var.strip(), prop.strip() or "name"))
    return {"nodes": nodes, "chain": chain, "returns": returns}


def traverse(G: nx.DiGraph, start: str, chain: List[tuple]) -> List[Dict]:
    """관계 타입 체인을 따라 다단 경로 탐색 (Cypher 패턴의 실행 엔진)."""
    frontier = [{"node": start, "path": [start], "rels": []}]
    for rel, direction in chain:
        next_frontier = []
        for f in frontier:
            cur = f["node"]
            if direction == "out":
                edges = [(cur, t, d) for _, t, d in G.out_edges(cur, data=True)]
            else:
                edges = [(s, cur, d) for s, _, d in G.in_edges(cur, data=True)]
            for s, t, d in edges:
                if d["relation"] == rel:
                    nxt = t if direction == "out" else s
                    next_frontier.append({
                        "node": nxt,
                        "path": f["path"] + [nxt],
                        "rels": f["rels"] + [(s, rel, t)],
                    })
        frontier = next_frontier
    return frontier


def run_cypher(G: nx.DiGraph, cypher: str) -> Dict:
    """Cypher 를 실행해 {header, rows, paths} 를 돌려준다."""
    q = parse_cypher(cypher)
    anchor = q["nodes"][0]["name"]
    if anchor is None:
        raise ValueError("첫 노드에 {name: '...'} 앵커가 필요합니다.")
    if anchor not in G:
        raise ValueError(f"그래프에 없는 노드입니다: '{anchor}'")

    paths = traverse(G, anchor, q["chain"])

    # (var:라벨) 제약 검사 — 타입이 어긋나는 경로는 탈락
    kept = [r for r in paths
            if all(node["label"] is None or G.nodes[n].get("type") == node["label"]
                   for node, n in zip(q["nodes"], r["path"]))]

    var_pos = {n["var"]: i for i, n in enumerate(q["nodes"]) if n["var"]}
    returns = q["returns"] or [(n["var"], "name") for n in q["nodes"] if n["var"]]

    rows, seen = [], set()
    for r in kept:
        row = []
        for v, p in returns:
            if v not in var_pos:
                raise ValueError(f"패턴에 없는 변수입니다: {v}")
            n = r["path"][var_pos[v]]
            row.append(n if p == "name" else G.nodes[n].get(p))
        if tuple(row) not in seen:
            seen.add(tuple(row))
            rows.append(tuple(row))
    return {"header": [f"{v}.{p}" for v, p in returns], "rows": rows, "paths": kept}


def print_table(res: Dict) -> None:
    """Neo4j 브라우저 비슷한 표로 출력."""
    header, rows = res["header"], res["rows"]
    width = lambda s: sum(2 if ord(c) > 0x1100 else 1 for c in str(s))
    widths = [max([width(h)] + [width(r[i]) for r in rows]) for i, h in enumerate(header)]
    pad = lambda s, w: str(s) + " " * (w - width(s))
    print("  " + " │ ".join(pad(h, w) for h, w in zip(header, widths)))
    print("  " + "─┼─".join("─" * w for w in widths))
    for r in rows:
        print("  " + " │ ".join(pad(c, w) for c, w in zip(r, widths)))
    print(f"  ({len(rows)} rows)")


print("Cypher 시뮬레이터 준비 완료")


In [ ]:
# Step 4: 자연어 질문 → Cypher 생성 → 검색 → LLM 답변

# LLM 에게 넘길 스키마. 그래프에서 직접 뽑으므로 데이터가 바뀌면 같이 따라간다.
node_labels = sorted({d.get("type", "?") for _, d in kg.nodes(data=True)})
edge_types = sorted({d["relation"] for _, _, d in kg.edges(data=True)})
schema_lines = sorted({
    f"(:{kg.nodes[s].get('type')})-[:{d['relation']}]->(:{kg.nodes[t].get('type')})"
    for s, t, d in kg.edges(data=True)
})

GRAPH_SCHEMA = (
    f"노드 라벨: {', '.join(node_labels)}\n"
    f"관계 타입: {', '.join(edge_types)}\n"
    "허용된 관계 방향:\n  " + "\n  ".join(schema_lines)
)

# Few-shot 예시 — 질문과 Cypher 를 짝지어 보여주면 형식이 안정된다.
FEW_SHOT = [
    ("SK그룹의 회장은 누구야?",
     "MATCH (g:그룹 {name: 'SK그룹'})-[:HAS_CHAIRMAN]->(p:인물) RETURN p.name"),
    ("삼성전자 CEO 는 어느 학교를 나왔어?",
     "MATCH (c:계열사 {name: '삼성전자'})-[:HAS_CEO]->(ceo:인물)"
     "-[:GRADUATED_FROM]->(s:학교) RETURN ceo.name, s.name"),
    # 역방향 예시 — 계열사→그룹 엣지를 그룹에서 거슬러 올라간다
    ("SK그룹에 속한 계열사들을 알려줘",
     "MATCH (g:그룹 {name: 'SK그룹'})<-[:SUBSIDIARY_OF]-(c:계열사) RETURN c.name"),
]

# 스키마를 (관계 → 허용된 (주어 라벨, 목적어 라벨)) 로 정리 — 방향 검증에 쓴다
SCHEMA_PAIRS = defaultdict(set)
for s_, t_, d_ in kg.edges(data=True):
    SCHEMA_PAIRS[d_["relation"]].add(
        (kg.nodes[s_].get("type"), kg.nodes[t_].get("type")))


def check_direction(cypher: str) -> List[str]:
    """생성된 Cypher 의 각 홉이 스키마의 관계 방향과 맞는지 검사."""
    try:
        q = parse_cypher(cypher)
    except ValueError:
        return []          # 파싱 자체가 안 되면 실행 단계에서 오류를 보여준다
    problems = []
    for i, (rel, direction) in enumerate(q["chain"]):
        if rel not in SCHEMA_PAIRS:
            problems.append(f"'{rel}' 은 스키마에 없는 관계 타입입니다.")
            continue
        a, b = q["nodes"][i]["label"], q["nodes"][i + 1]["label"]
        if a is None or b is None:
            continue                      # 라벨이 없으면 판정 불가
        pair = (a, b) if direction == "out" else (b, a)
        if pair not in SCHEMA_PAIRS[rel]:
            ok = ", ".join(f"(:{x})-[:{rel}]->(:{y})"
                           for x, y in sorted(SCHEMA_PAIRS[rel]))
            wrote = (f"(:{a})-[:{rel}]->(:{b})" if direction == "out"
                     else f"(:{a})<-[:{rel}]-(:{b})")
            problems.append(
                f"{wrote} 는 스키마와 방향이 다릅니다. 허용: {ok}")
    return problems


def repair_cypher(cypher: str):
    """스키마를 기준으로 관계 화살표 방향을 결정적으로 교정.

    LLM 에게 다시 시켜도 같은 실수를 반복하는 경우가 많다.
    방향은 스키마만 보면 유일하게 정해지므로 코드가 직접 고친다.
    """
    text = " ".join(cypher.split())
    m = re.search(r"(MATCH\s+)(.*?)(\s+RETURN\s+.*)?$", text, re.I)
    if not m:
        return cypher, []
    head, pattern, tail = m.group(1), m.group(2), m.group(3) or ""

    nodes, spans = [], []
    for nm in NODE_RE.finditer(pattern):
        nodes.append({"label": nm.group(2)})
        spans.append(nm.span())

    out, notes, prev = [], [], 0
    for i in range(len(nodes) - 1):
        conn = pattern[spans[i][1]:spans[i + 1][0]]
        rm = REL_RE.search(conn)
        rel = rm.group(1) if rm else None
        a, b = nodes[i]["label"], nodes[i + 1]["label"]

        want = None
        if rel in SCHEMA_PAIRS and a and b:
            if (a, b) in SCHEMA_PAIRS[rel]:
                want = "out"
            elif (b, a) in SCHEMA_PAIRS[rel]:
                want = "in"

        fixed = conn
        if want == "out":
            fixed = f"-[:{rel}]->"
        elif want == "in":
            fixed = f"<-[:{rel}]-"
        if fixed.strip() != conn.strip():
            notes.append(f"(:{a}){conn.strip()}(:{b})  →  (:{a}){fixed}(:{b})")

        out.append(pattern[prev:spans[i][1]])
        out.append(fixed)
        prev = spans[i + 1][0]
    out.append(pattern[prev:])
    return head + "".join(out) + tail, notes

TEXT2CYPHER_PROMPT = """당신은 한국어 질문을 Cypher 로 변환하는 변환기입니다.

그래프 스키마:
{schema}

규칙:
1. 스키마에 있는 라벨·관계 타입만 사용하세요.
2. 관계 방향을 스키마대로 지키세요. 거슬러 올라갈 때는 <- 를 쓰세요.
3. 시작 노드에는 반드시 {{name: '...'}} 를 붙이세요.
4. 선형 경로 패턴 하나만 쓰고, WHERE·가변길이(*)·집계 함수는 쓰지 마세요.
5. Cypher 문 한 줄만 출력하세요. 설명·마크다운 코드펜스 금지.

예시:
{examples}

질문: {question}
Cypher:"""


def nl_to_cypher(question: str) -> str:
    """자연어 질문 → Cypher (LLM 호출)."""
    examples = "\n".join(f"질문: {q}\nCypher: {c}\n" for q, c in FEW_SHOT)
    prompt = TEXT2CYPHER_PROMPT.format(
        schema=GRAPH_SCHEMA, examples=examples, question=question)
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    text = response.choices[0].message.content.strip()
    return re.sub(r"^```(?:cypher)?|```$", "", text, flags=re.M).strip()


print("=" * 60)
print("[LLM 에게 전달되는 스키마]")
print("=" * 60)
print(GRAPH_SCHEMA)


QUESTIONS = [
    "이재용이 회장인 그룹의 모든 계열사 CEO 들이 어느 학교를 나왔는지 알려줘.",
    "현대자동차그룹 계열사들의 CEO 는 각각 누구야?",
    "최태원은 어느 학교를 나왔어?",
]

GRAPH_RAG_PROMPT = """다음 지식 그래프 정보를 참고해 질문에 답하세요.

지식 그래프 컨텍스트:
{context}

질문: {question}

컨텍스트에 없는 내용은 답에 넣지 마세요."""


def graph_rag_answer(question: str, triples: List[str]) -> str:
    """검색된 트리플만 근거로 답변 생성."""
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user",
                   "content": GRAPH_RAG_PROMPT.format(
                       context="\n".join(triples), question=question)}],
        temperature=0,
    )
    return response.choices[0].message.content


def ask(question: str) -> None:
    """질문 → Cypher → 실행 → 답변. 그래프 RAG 한 바퀴를 끝까지 돈다."""
    print("=" * 60)
    print(f"질문: {question}")
    print("=" * 60)

    # ① 자연어 → Cypher (스키마 방향이 어긋나면 피드백을 주고 1회 재시도)
    cypher = nl_to_cypher(question)
    print(f"\n① 생성된 Cypher\n     {cypher}")

    # 방향은 스키마만 보면 유일하게 정해진다 → 코드가 먼저 교정한다
    cypher, notes = repair_cypher(cypher)
    if notes:
        print("\n   ⚠ 스키마와 방향이 달라 자동 교정했습니다")
        for n in notes:
            print(f"     · {n}")
        print(f"\n   교정된 Cypher\n     {cypher}")

    for p in check_direction(cypher):
        print(f"   ⚠ 교정 후에도 스키마와 맞지 않습니다 — {p}")

    # ② Cypher 실행
    print("\n② 실행 결과")
    try:
        res = run_cypher(kg, cypher)
    except ValueError as e:
        # 스키마를 벗어난 Cypher 는 여기서 잡힌다 — 감추지 않고 보여준다.
        print(f"   ⚠ 실행할 수 없는 Cypher 입니다 — {e}")
        return
    print_table(res)
    if not res["rows"]:
        print("   → 0행이므로 답변을 생성하지 않습니다 (근거 없는 서술 방지).")
        return

    # ③ 검색 결과를 트리플로 펴서 컨텍스트 구성
    triples = sorted({f"({s}) --[{rel}]--> ({t})"
                      for r in res["paths"] for s, rel, t in r["rels"]})
    print(f"\n③ 컨텍스트 트리플 {len(triples)}개")
    for t in triples:
        print(f"     {t}")

    # ④ 트리플만 근거로 답변 생성
    print("\n④ 최종 답변")
    try:
        print("   " + graph_rag_answer(question, triples).replace("\n", "\n   "))
    except Exception as e:
        print(f"   ⚠ 답변 생성 생략 ({type(e).__name__})")


for q in QUESTIONS:
    print()
    try:
        ask(q)
    except Exception as e:
        print(f"⚠ LLM 호출 생략 ({type(e).__name__}) — 이 셀은 API 키가 필요합니다.")
        break


---

## 정리

### 핵심 요약

| 항목 | 벡터 RAG | 그래프 RAG |
|------|----------|------------|
| 데이터 구조 | 벡터 임베딩 | 지식 그래프 (트리플) |
| 검색 방식 | 유사도 검색 | 그래프 탐색 (BFS/DFS) |
| 강점 | 의미적 유사성 | 관계 추론, 멀티홉 |
| 약점 | 관계 정보 손실 | 그래프 구축 비용 |
| 적합한 질문 | 단순 사실 질문 | 관계/추론 질문 |

### 다음 단계

- 온톨로지 RAG: 도메인 지식을 체계적으로 정의하여 더 강력한 추론을 수행
- 하이브리드 RAG: 벡터 검색 + 그래프 검색을 결합한 최적의 접근
- Microsoft GraphRAG: 커뮤니티 기반 글로벌/로컬 검색

In [ ]:
print("=" * 60)
print("그래프 RAG 실습 완료!")
print("=" * 60)
print()
print("학습 내용 정리:")
print("  1. 지식 그래프 구축: 텍스트 → LLM 추출 → 노드/엣지")
print("  2. Cypher 시뮬레이터: 경로 패턴 파싱 → NetworkX 탐색 (Neo4j 불필요)")
print("  3. Text-to-Cypher: 자연어 질문을 LLM 이 Cypher 로 번역")
print("  4. 검색된 트리플만 근거로 LLM 답변 생성")
